# Seasonal Adjustment (Monthly, UK Calendar) — Module & Diagnostics
This notebook demonstrates the seasonal adjustment utility in `seasonal_adjustment.py`
with monthly data, UK-specific working-day calendar (bank holidays), trading-day and Easter effects,
and an optional X-13 path for closer alignment with JDemetra+.


In [ ]:
import pandas as pd
import numpy as np
from seasonal_adjustment import seasonal_adjust, plot_sa

# Optional: if you intend to use X-13ARIMA-SEATS, ensure statsmodels can find the X-13 binary.
# In statsmodels, this typically works out of the box if the binary is on PATH.
# Otherwise, you can configure the path using statsmodels.tsa.x13.x13_arima_analysis arguments.


## Load your monthly data
Provide a monthly pandas Series `y` indexed by a fixed monthly DatetimeIndex (e.g., end-of-month).
Replace the synthetic example below with your actual data (CSV/Excel read).


In [ ]:
# Example: synthetic monthly series with multiplicative seasonality
idx = pd.date_range('2015-01-31', periods=132, freq='M')
rng = np.random.default_rng(42)
season = np.tile([0.90, 0.92, 0.95, 0.98, 1.02, 1.05, 1.08, 1.07, 1.03, 1.00, 0.97, 0.93], 11)
trend = 100 * (1 + 0.003) ** np.arange(len(idx))
irreg = rng.normal(0, 0.8, len(idx))
y = pd.Series(trend * season * np.exp(irreg/50), index=idx, name='y')
y.head()


## Run seasonal adjustment (UK working days, trading-day, length-of-month, Easter)
Set `method='auto'` to prefer X-13 when available; otherwise STL fallback.


In [ ]:
res = seasonal_adjust(
    y,
    method='auto',            # try X-13, else STL
    multiplicative=True,
    use_trading_day=True,
    use_length_effect=True,
    use_easter=True,
    easter_k=8,
    use_uk_working_days=True,
    outlier_method='hampel'
)
res.diagnostics


In [ ]:
_ = plot_sa(y, res, title='Seasonal Adjustment (UK calendar; auto method)')


## Components
Access the adjusted series and components below.


In [ ]:
sa = res.sa
seasonal = res.seasonal
trend = res.trend
irregular = res.irregular
sa.head(), seasonal.head()


### Notes on matching JDemetra+ more closely
- Install and let statsmodels find the **X-13ARIMA-SEATS** binary, then run with `method='x13'`.
- If your series uses country-specific holiday exceptions (e.g., special jubilees), add them via
  the `extra_uk_holidays` parameter.
- For diagnostics, `res.diagnostics['auto_arima_summary']` (if `pmdarima` is installed) shows the
  identified ARIMA model on the pre-adjusted working series.


In [ ]:
import warnings
from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.seasonal import STL

# Optional: X-13ARIMA-SEATS (will be used if available)
try:
    from statsmodels.tsa.x13 import x13_arima_analysis
    _HAS_X13 = True
except Exception:
    _HAS_X13 = False

# Optional: Easter from pandas (for calendar effects)
try:
    from pandas.tseries.holiday import Easter
    _HAS_EASTER = True
except Exception:
    _HAS_EASTER = False


@dataclass
class SAResult:
    sa: pd.Series                     # Seasonally adjusted series
    seasonal: pd.Series               # Seasonal component
    trend: pd.Series                  # Trend from STL
    irregular: pd.Series              # Remainder from STL
    model: str                        # 'stl' or 'x13'
    used_exog: Optional[pd.DataFrame] # Calendar regressors actually used
    coef: Optional[pd.Series]         # OLS coefficients for exog (if used)
    diagnostics: Dict[str, Any]       # Misc diagnostics


def _infer_periods(y: pd.Series, seasonal_periods: Optional[int]) -> int:
    """Infer seasonal periods from index or use supplied value."""
    if seasonal_periods is not None:
        return seasonal_periods

    # Try to infer from freq
    freq = getattr(y.index, "freqstr", None) or pd.infer_freq(y.index)
    if freq is None:
        raise ValueError("Cannot infer frequency. Please set `seasonal_periods` explicitly.")
    f = freq.upper()

    # Common cases
    if f.startswith("M"):      # Monthly
        return 12
    if f.startswith("Q"):      # Quarterly
        return 4
    if f.startswith("W"):      # Weekly (not ideal for STL with TD regressors)
        return 52
    if f.startswith("D"):      # Daily
        return 7

    # Fallback
    raise ValueError(f"Unsupported/inferable frequency '{freq}'. Provide `seasonal_periods`.")


def _count_weekdays_in_period(index: pd.DatetimeIndex) -> pd.DataFrame:
    """
    For each period (month or quarter based on index.freq), count Mon..Sat minus Sundays.
    Returns 6 regressors (Mon..Sat) following X-13 6-day TD style.
    """
    if index.freq is None:
        # Attempt to set frequency (won't fill gaps)
        inferred = pd.infer_freq(index)
        if inferred is None:
            raise ValueError("Index frequency is missing. Please set a fixed frequency.")
        index = index.asfreq(inferred)

    # Decide grouping by period
    if index.freqstr.upper().startswith("M"):
        grouper = index.to_period("M")
    elif index.freqstr.upper().startswith("Q"):
        grouper = index.to_period("Q")
    else:
        raise ValueError("Trading-day regressors are only implemented for monthly/quarterly.")

    # For each period, count weekdays
    rows = []
    for p in pd.period_range(grouper.min(), grouper.max(), freq=grouper.freq):
        # All days in this period
        if p.freqstr == "M":
            start = p.to_timestamp(how="start")
            end = p.to_timestamp(how="end")
        else:  # Quarterly
            start = p.start_time
            end = p.end_time

        # Build daily date range
        dr = pd.date_range(start, end, freq="D")
        # Weekday: Monday=0 ... Sunday=6
        counts = dr.weekday.value_counts().reindex(range(7), fill_value=0)
        # 6 TD regressors: Mon..Sat minus Sunday
        td = counts.loc[0:5].values - counts.loc[6]
        rows.append((p.to_timestamp(how="end"), *td, counts.sum()))

    df = pd.DataFrame(rows, columns=["period_end", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "days"])
    df.set_index("period_end", inplace=True)

    # Center TD regressors to sum to ~0 over sample (common practice)
    td_cols = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
    df[td_cols] = df[td_cols] - df[td_cols].mean(axis=0)

    # Length-of-period effect (centered)
    df["len"] = df["days"] - df["days"].mean()
    df.drop(columns=["days"], inplace=True)
    return df.reindex(index)


def _easter_regressor(index: pd.DatetimeIndex, k: int = 8) -> pd.Series:
    """
    Fractional Easter effect over [-k, 0] days before Easter Sunday within each period.
    Matches common practice to distribute effect into the period containing Easter.
    """
    if not _HAS_EASTER:
        raise ImportError("pandas.tseries.holiday.Easter not available for Easter regression.")

    # Determine grouping (month or quarter)
    if index.freqstr is None:
        inferred = pd.infer_freq(index)
        if inferred is None:
            raise ValueError("Index frequency is missing. Please set a fixed frequency.")
        index = index.asfreq(inferred)

    if index.freqstr.upper().startswith("M"):
        to_period = "M"
    elif index.freqstr.upper().startswith("Q"):
        to_period = "Q"
    else:
        raise ValueError("Easter regressor is implemented for monthly/quarterly data.")

    periods = index.to_period(to_period)
    out = pd.Series(0.0, index=index)

    for p in periods.unique():
        # Period boundaries
        if to_period == "M":
            start = p.to_timestamp(how="start")
            end = p.to_timestamp(how="end")
        else:
            start = p.start_time
            end = p.end_time

        # Easter Sunday for the year
        easter_date = pd.Timestamp(Easter()(p.year))
        window_start = easter_date - pd.Timedelta(days=k)
        window_end = easter_date  # inclusive of Easter Sunday

        # Overlap with this period
        overlap_start = max(start, window_start)
        overlap_end = min(end, window_end)

        if overlap_start <= overlap_end:
            # Fraction of k+1 days that fall in this period
            days_in_window = (window_end - window_start).days + 1
            days_in_period = (overlap_end - overlap_start).days + 1
            frac = days_in_period / max(days_in_window, 1)
        else:
            frac = 0.0

        out.loc[index[(index >= start) & (index <= end)]] = frac

    # Center the regressor
    out = out - out.mean()
    return out


def _hampel_filter(x: pd.Series, window: int = 7, n_sigmas: float = 3.0) -> Tuple[pd.Series, pd.Series]:
    """
    Hampel filter to flag/replace outliers using rolling median & MAD.
    Returns (cleaned_series, outlier_mask).
    """
    x = x.astype("float64")
    med = x.rolling(window, center=True, min_periods=1).median()
    mad = (np.abs(x - med)).rolling(window, center=True, min_periods=1).median()
    # 1.4826 approximates std from MAD under normality
    threshold = n_sigmas * 1.4826 * mad.replace(0, np.nan)
    outliers = (np.abs(x - med) > threshold).fillna(False)
    cleaned = x.copy()
    cleaned[outliers] = med[outliers]
    return cleaned, outliers


def seasonal_adjust(
    y: pd.Series,
    seasonal_periods: Optional[int] = None,
    *,
    method: str = "auto",           # 'auto' | 'stl' | 'x13'
    multiplicative: bool = True,
    use_log: Optional[bool] = None, # If None: auto (True if y>0 and multiplicative)
    robust_stl: bool = True,
    outlier_method: str = "hampel", # 'none' | 'hampel'
    outlier_window: int = 7,
    outlier_sigmas: float = 3.0,
    use_trading_day: bool = True,
    use_length_effect: bool = True,
    use_easter: bool = True,
    easter_k: int = 8,
    extra_exog: Optional[pd.DataFrame] = None,
) -> SAResult:
    """
    Seasonally adjust a monthly/quarterly (also supports other freqs for STL-only) time series.

    Parameters
    ----------
    y : pd.Series
        Time series with a fixed pandas DatetimeIndex (freq set or inferable).
    seasonal_periods : int, optional
        Seasonal period (12 for monthly, 4 for quarterly). If None, inferred from index freq.
    method : {'auto','stl','x13'}
        'x13' tries X-13ARIMA-SEATS if available; otherwise falls back to STL.
        'auto' uses 'x13' if available, else 'stl'.
    multiplicative : bool
        Interpret seasonality multiplicatively (common for economic indicators).
    use_log : bool or None
        If None, auto-enable log when y>0 and multiplicative=True.
    robust_stl : bool
        Use robust STL to lessen outlier impact.
    outlier_method : {'none','hampel'}
        Pre-STL outlier mitigation on the working series.
    outlier_window : int
        Window size for Hampel filter.
    outlier_sigmas : float
        Sigma threshold for Hampel filter.
    use_trading_day : bool
        Include 6-variable TD regressors (Mon..Sat - Sun) for monthly/quarterly.
    use_length_effect : bool
        Include centered length-of-period regressor.
    use_easter : bool
        Include centered Easter regressor (requires pandas.tseries.holiday.Easter).
    easter_k : int
        Pre-Easter window length, e.g., 8 days.
    extra_exog : pd.DataFrame
        Additional user-supplied regressors aligned to y.index.

    Returns
    -------
    SAResult
        Contains seasonally adjusted series, components, exog info, and diagnostics.
    """
    if not isinstance(y.index, pd.DatetimeIndex):
        raise TypeError("`y` must have a pandas DatetimeIndex.")
    y = y.sort_index().asfreq(pd.infer_freq(y.index) or y.index.freq)

    # Determine seasonal periods
    s = _infer_periods(y, seasonal_periods)

    # Decide log-transform
    if use_log is None:
        use_log = multiplicative and (y.min() > 0)

    # Build calendar regressors (monthly/quarterly only)
    exog_parts = []
    exog_notes = []
    if y.index.freqstr and y.index.freqstr.upper().startswith(("M", "Q")):
        if use_trading_day or use_length_effect:
            td_len = _count_weekdays_in_period(y.index)
            cols = []
            if use_trading_day:
                cols += ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
                exog_notes.append("trading_day")
            if use_length_effect:
                cols += ["len"]
                exog_notes.append("length")
            exog_parts.append(td_len[cols])

        if use_easter:
            if not _HAS_EASTER:
                warnings.warn("Easter regressor requested but pandas Easter not available; skipped.")
            else:
                exog_parts.append(_easter_regressor(y.index, k=easter_k).to_frame("Easter"))
                exog_notes.append("easter")
    elif any([use_trading_day, use_length_effect, use_easter]):
        warnings.warn("Calendar regressors are only implemented for monthly/quarterly; skipping.")

    if extra_exog is not None:
        extra_exog = extra_exog.reindex(y.index)
        exog_parts.append(extra_exog)
        exog_notes.append("extra_exog")

    X = None
    if exog_parts:
        X = pd.concat(exog_parts, axis=1).fillna(0.0)
        # De-mean (common for deterministic regressors)
        X = X - X.mean()

    # Prepare working series (remove deterministic effects by OLS)
    if use_log:
        if (y <= 0).any():
            raise ValueError("Log transform requested but series contains non-positive values.")
        wy = np.log(y.copy())
    else:
        wy = y.astype("float64").copy()

    # OLS to remove exogenous calendar effects
    coef = None
    if X is not None and X.shape[1] > 0:
        X_ols = sm.add_constant(X)  # include intercept
        ols = sm.OLS(wy.values, X_ols.values, missing='drop').fit()
        coef = pd.Series(ols.params, index=X_ols.columns, name="coef")
        fitted_det = pd.Series(ols.fittedvalues, index=wy.index, name="deterministic_fit")
        resid = wy - fitted_det
    else:
        fitted_det = pd.Series(0.0, index=wy.index, name="deterministic_fit")
        resid = wy.copy()

    # Outlier mitigation on residual working series (pre-STL)
    outlier_mask = pd.Series(False, index=y.index)
    if outlier_method == "hampel":
        resid, outlier_mask = _hampel_filter(resid, window=outlier_window, n_sigmas=outlier_sigmas)

    # Try X-13 if requested/available; else STL
    chosen_method = None
    if method in ("auto", "x13") and _HAS_X13:
        try:
            # X-13 can take exogenous regressors if passed via 'x12path' wrapper settings,
            # but statsmodels provides a high-level interface without direct exog.
            # Here we run a plain X-13 and fallback to STL if it fails.
            # Note: X-13 expects no NaNs and a fixed freq.
            x13_res = x13_arima_analysis(wy.dropna())
            seasonal = x13_res.seasadj - wy  # seasadj = wy - seasonal
            sa_w = x13_res.seasadj
            chosen_method = "x13"
            # In X-13 path, we do not incorporate exog explicitly (keep OLS det. separately)
            trend = pd.Series(np.nan, index=wy.index)
            irregular = wy - seasonal - fitted_det  # placeholder residual
        except Exception:
            warnings.warn("X-13 failed or not properly installed. Falling back to STL.")
            chosen_method = None

    if chosen_method is None:
        # STL decomposition on resid
        stl = STL(
            resid,
            period=s,
            robust=robust_stl,
            seasonal=13,    # default Loess span; tune as needed
            trend=max(7, s + 1),
            low_pass=None
        ).fit()
        seasonal = stl.seasonal
        trend = stl.trend
        irregular = stl.resid
        sa_w = resid - seasonal
        chosen_method = "stl"

    # Recombine deterministic part and undo transforms
    sa = sa_w + fitted_det  # back to wy-scale
    if use_log:
        # wy = log(y), sa is log-scale seasonally adjusted -> exponentiate
        sa = np.exp(sa)
        seasonal_comp = np.exp(seasonal) if multiplicative else seasonal
    else:
        seasonal_comp = seasonal

    # Diagnostics
    diag = {
        "method": chosen_method,
        "seasonal_periods": s,
        "used_log": use_log,
        "multiplicative": multiplicative,
        "outliers_flagged": int(outlier_mask.sum()) if outlier_method == "hampel" else 0,
        "exog_included": exog_notes,
        "index_freq": y.index.freqstr,
    }

    # Return a full set of components aligned to y.index (STL produces matched index)
    return SAResult(
        sa=sa.rename("sa"),
        seasonal=seasonal_comp.rename("seasonal"),
        trend=trend.rename("trend"),
        irregular=irregular.rename("irregular"),
        model=chosen_method,
        used_exog=(X if X is not None and X.shape[1] > 0 else None),
        coef=coef,
        diagnostics=diag,
    )

In [ ]:
#Example usage
# Example: monthly CPI-type index with multiplicative seasonality
# y must be a DatetimeIndex with fixed monthly frequency.

# Suppose you have a pandas Series `y` indexed monthly:
# y = pd.Series(..., index=pd.date_range("2015-01-01", periods=120, freq="M"))

res = seasonal_adjust(
    y,
    method="auto",            # 'auto' will try X-13 if available, else STL
    multiplicative=True,
    robust_stl=True,
    use_trading_day=True,
    use_length_effect=True,
    use_easter=True,
    easter_k=8,
    outlier_method="hampel",
)

print(res.diagnostics)
sa = res.sa                     # Seasonally adjusted series
seasonal = res.seasonal         # Seasonal factor
trend = res.trend               # Trend from STL
irregular = res.irregular       # Irregular component
coef = res.coef                 # Calendar regression coefficients

# Plot (optional)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10,5))
y.plot(ax=ax, alpha=0.5, label="Original")
sa.plot(ax=ax, label="Seasonally Adjusted")
ax.legend()
ax.set_title("Seasonal Adjustment (STL fallback if X-13 unavailable)")
plt.show()

In [ ]:
from seasonal_adjustment import seasonal_adjust, plot_sa

res = seasonal_adjust(
    y,                         # monthly pandas Series with DatetimeIndex (fixed freq)
    method="auto",             # 'auto' tries X-13, else STL
    multiplicative=True,       # log-transform if data > 0
    use_trading_day=True,
    use_length_effect=True,
    use_easter=True,
    easter_k=8,
    use_uk_working_days=True,  # subtract UK bank holidays in weekday counts
    extra_uk_holidays=[        # optional one-offs for exact replication
        # pd.Timestamp("2022-06-02"), pd.Timestamp("2022-09-19"), ...
    ],
    outlier_method="hampel",   # or 'none'
)

# Results
sa        = res.sa           # seasonally adjusted series (original scale)
seasonal  = res.seasonal     # seasonal factor (mult. if log transform applied)
trend     = res.trend        # STL trend (NaN in X-13 path)
irregular = res.irregular    # STL residual (NaN in X-13 path)
diag      = res.diagnostics  # method, periods, log used, # outliers, exogs, auto-arima summary

In [ ]:
import pandas as pd
from seasonal_adjustment import seasonal_adjust, plot_sa

# Replace with your monthly series (end-of-month index recommended)
# y = pd.read_csv("my_monthly_series.csv", parse_dates=["date"], index_col="date")["value"].asfreq("M")

res = seasonal_adjust(
    y,
    method="auto",           # will use X-13 if available
    multiplicative=True,
    use_trading_day=True,
    use_length_effect=True,
    use_easter=True,
    easter_k=8,
    use_uk_working_days=True,
    # Exact replication? Add special one-off UK bank holidays for your sample period:
    # extra_uk_holidays=[pd.Timestamp("2022-06-02"), pd.Timestamp("2022-06-03"), pd.Timestamp("2022-09-19")],
    outlier_method="hampel",
)

print(res.diagnostics)
_ = plot_sa(y, res, title="Seasonal Adjustment (UK calendar; JDemetra-like)")